# Proyecto RappiPlus: de datos a decisiones de negocio

**Introducción**


El objetivo de este proyecto es evaluar el desempeño del servicio **RappiPlus** para apoyar **decisiones de negocio basadas en datos**.

Se trabajan con múltiples datasets del negocio:

- **rappiplus_orders_raw.csv** → información de pedidos, precios, descuentos y revenue  
- **rappiplus_catalog.csv** → costos de productos, categorías y proveedores  
- **rappiplus_marketing_spend.csv** → inversión en marketing por canal y país  
- **events / users / user_activity (SQL)** → comportamiento del usuario dentro de la plataforma  
- **experiment_checkout_ui.csv** → resultados de un experimento A/B en el checkout  

El análisis sigue una lógica clara y progresiva:

1. 🔍 Evaluar si podemos confiar en los datos (calidad de datos en Python) 

2. 💰 Analizar si el negocio es rentable (revenue, costos y profit)  

3. 🛒 Entender dónde se pierden los usuarios (funnel de conversión)  

4. 🔁 Evaluar si los usuarios regresan (retención por cohortes)  

5. 🧪 Validar si los cambios generan impacto (test estadístico)  

6. 📊 Comunicar los resultados (dashboard en BI)  

A lo largo del proyecto, se transforman datos en insights para responder preguntas clave del negocio y proponer **recomendaciones accionables**.

---

## 🔹 Paso 1: Cargar y validar la calidad de los datos

---

### 1.1 Carga de datos y vista rápida

**🎯 Objetivo:** Familiarizarte con la estructura de los datasets del negocio antes de analizarlos.

**Instrucciones:**

- Importa las librerías necesarias
- Carga los archivos:
  - `rappiplus_orders_raw.csv`
  - `rappiplus_catalog.csv`
  - `rappiplus_marketing_spend.csv`
- Guarda los DataFrames en:
  - `orders`, `catalog`, `marketing`
- Explora cada dataset.

---

In [40]:
# importar librerías
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [41]:
# cargar archivos
orders = pd.read_csv('datasets/rappiplus_orders_raw.csv') # tu código aquí
catalog = pd.read_csv('datasets/rappiplus_catalog.csv') # tu código aquí
marketing = pd.read_csv('datasets/rappiplus_marketing_spend.csv') # tu código aquí

In [42]:
# explorar datasets
print("Orders:", orders.shape)
print("Catalog:", catalog.shape)
print("Marketing:", marketing.shape)

Orders: (25100, 12)
Catalog: (7, 4)
Marketing: (1620, 5)


In [43]:
# explorar datasets
# tu código aquí
orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25100 entries, 0 to 25099
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id_pedido           25100 non-null  object 
 1   id_usuario          25100 non-null  object 
 2   fecha_hora_pedido   25100 non-null  object 
 3   pais                24800 non-null  object 
 4   dispositivo         25080 non-null  object 
 5   fuente_referencia   25070 non-null  object 
 6   nombre_producto     25070 non-null  object 
 7   categoria_producto  25020 non-null  object 
 8   cantidad            25050 non-null  float64
 9   precio_unitario     25050 non-null  float64
 10  monto_descuento     25050 non-null  float64
 11  monto_total         25100 non-null  float64
dtypes: float64(4), object(8)
memory usage: 2.3+ MB


In [44]:
# explorar datasets
# tu código aquí
catalog.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   nombre_producto     7 non-null      object 
 1   categoria_producto  7 non-null      object 
 2   costo_unitario      7 non-null      float64
 3   proveedor           7 non-null      object 
dtypes: float64(1), object(3)
memory usage: 352.0+ bytes


In [45]:
marketing.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1620 entries, 0 to 1619
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   fecha       1620 non-null   object 
 1   pais        1620 non-null   object 
 2   id_campaña  1620 non-null   object 
 3   canal       1519 non-null   object 
 4   gasto       1620 non-null   float64
dtypes: float64(1), object(4)
memory usage: 63.4+ KB


In [46]:
# Estadísticas descriptivas
orders.describe()

,cantidad,precio_unitario,monto_descuento,monto_total
count,25050.000000,25050.000000,25050.000000,2.510000e+04
mean,7.092735,259.305549,4.500798,2.072680e+03
std,296.277003,138.726461,5.223010,9.894995e+04
min,-2.000000,20.030000,0.000000,-4.926500e+02
25%,1.000000,138.377500,0.000000,1.805075e+02
50%,2.000000,258.715000,0.000000,3.417500e+02
75%,2.000000,380.332500,10.000000,5.185800e+02
max,20000.000000,499.960000,15.000000,8.840200e+06


In [47]:
# Estadísticas descriptivas
catalog.describe()

,costo_unitario
count,7.000000
mean,102.252857
std,111.011563
min,10.120000
25%,16.905000
50%,25.210000
75%,182.975000
max,280.680000


In [48]:
# Estadísticas descriptivas
marketing.describe()

,gasto
count,1620.00000
mean,1772.74292
std,734.43294
min,501.11000
25%,1128.03000
50%,1782.42500
75%,2420.68500
max,2999.36000


In [49]:
# Convertir fecha a datetime
orders['fecha_hora_pedido'] = pd.to_datetime(orders['fecha_hora_pedido'])

# Verificar que quedó bien
print(orders['fecha_hora_pedido'].dtype)

datetime64[ns]


In [50]:
# Esto te mostrará las primeras 3 filas reales de cada tabla
print("--- Muestra de Órdenes ---")
display(orders.head(3))

print("\n--- Muestra de Catálogo ---")
display(catalog.head(3))

print("\n--- Muestra de Marketing ---")
display(marketing.head(3))

--- Muestra de Órdenes ---


,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
0,order_0,user_6993,2025-05-22,Argentina,desktop,organic,Jacket-Winter-M,Moda,2.0,332.69,0.0,665.37
1,order_1,user_1329,2025-06-15,Mexico,desktop,paid_search,Tablet-Standard-64GB,Electronica,1.0,176.86,5.0,171.86
2,order_2,user_3194,2025-05-02,Argentina,desktop,social,Blender-XL-Red,Hogar,2.0,102.99,10.0,195.99



--- Muestra de Catálogo ---


,nombre_producto,categoria_producto,costo_unitario,proveedor
0,Laptop-Gaming-16GB,Electrónica,280.68,"Fuller, Pena and Myers"
1,Phone-Pro-128GB,Electrónica,10.12,King Ltd
2,Tablet-Standard-64GB,Electrónica,25.21,Bowers LLC



--- Muestra de Marketing ---


,fecha,pais,id_campaña,canal,gasto
0,2025-01-01,Mexico,organic_Mexico,organic,2446.25
1,2025-01-01,Mexico,paid_search_Mexico,paid_search,2704.34
2,2025-01-01,Mexico,social_Mexico,social,2045.01


---

### Revisión y calidad de datos

**🎯 Objetivo:** Detectar y corregir problemas en los datos que puedan afectar el análisis de revenue, costos y rentabilidad.

Se revisan los 3 datasets
- Validar y convertir fechas al formato correcto  
- Revisar variables numéricas (sin negativos o ceros inválidos)  
- Verificar consistencia de montos  
- Eliminar duplicados  
- Revisar variables categóricas 

---

In [51]:
# tu código aquí
# Convertir columnas a formato fecha
orders['fecha_hora_pedido'] = pd.to_datetime(orders['fecha_hora_pedido'])
marketing['fecha'] = pd.to_datetime(marketing['fecha'])

# Verificar que el cambio se aplicó (deben decir datetime64[ns])
print("Tipo de fecha en orders:", orders['fecha_hora_pedido'].dtype)
print("Tipo de fecha en marketing:", marketing['fecha'].dtype)

Tipo de fecha en orders: datetime64[ns]
Tipo de fecha en marketing: datetime64[ns]


In [52]:

print("--- Auditoría Numérica de Órdenes ---")
print(orders[['cantidad', 'precio_unitario', 'monto_descuento', 'monto_total']].describe().loc[['min', 'max']])

print("\n--- Auditoría Numérica de Catálogo ---")
print(catalog[['costo_unitario']].describe().loc[['min', 'max']])

print("\n--- Auditoría Numérica de Marketing ---")
print(marketing[['gasto']].describe().loc[['min', 'max']])


--- Auditoría Numérica de Órdenes ---
     cantidad  precio_unitario  monto_descuento  monto_total
min      -2.0            20.03              0.0      -492.65
max   20000.0           499.96             15.0   8840200.00

--- Auditoría Numérica de Catálogo ---
     costo_unitario
min           10.12
max          280.68

--- Auditoría Numérica de Marketing ---
       gasto
min   501.11
max  2999.36


In [53]:
# Guardamos cuántas filas teníamos originalmente
filas_originales = len(orders)

# Aplicamos el filtro para eliminar cantidades o montos negativos/ceros
orders = orders[(orders['cantidad'] > 0) & (orders['monto_total'] > 0)]

# Verificamos cuántas filas se eliminaron
filas_limpias = len(orders)
print(f"Filas eliminadas por valores negativos: {filas_originales - filas_limpias}")

# Validamos que los mínimos ya no sean negativos
print("\n--- Nuevos mínimos en órdenes ---")
print(orders[['cantidad', 'monto_total']].min())

Filas eliminadas por valores negativos: 54

--- Nuevos mínimos en órdenes ---
cantidad       1.00
monto_total    5.24
dtype: float64


In [54]:
# 1. Calculamos el monto esperado según la fórmula teórica
monto_calculado = (orders['cantidad'] * orders['precio_unitario']) - orders['monto_descuento']

# 2. Buscamos registros donde la diferencia absoluta sea mayor a 0.01 (para ignorar centavos por redondeo)
inconsistencias = orders[abs(orders['monto_total'] - monto_calculado) > 0.01]

print(f"Número de filas con montos inconsistentes: {len(inconsistencias)}")

# Si hay inconsistencias, vemos una pequeña muestra de ellas
if len(inconsistencias) > 0:
    print("\nMuestra de las inconsistencias encontradas:")
    display(inconsistencias[['cantidad', 'precio_unitario', 'monto_descuento', 'monto_total']].head())

Número de filas con montos inconsistentes: 1146

Muestra de las inconsistencias encontradas:


,cantidad,precio_unitario,monto_descuento,monto_total
2,2.0,102.99,10.0,195.99
24,2.0,117.99,0.0,235.99
35,2.0,467.34,5.0,929.69
41,2.0,37.77,10.0,65.53
136,2.0,211.05,15.0,407.09


In [55]:
# Recalculamos el monto_total para eliminar las pequeñas variaciones de redondeo
orders['monto_total'] = (orders['cantidad'] * orders['precio_unitario']) - orders['monto_descuento']

# Volvemos a validar si quedan inconsistencias
monto_calculado_check = (orders['cantidad'] * orders['precio_unitario']) - orders['monto_descuento']
inconsistencias_restantes = orders[abs(orders['monto_total'] - monto_calculado_check) > 0.01]

print(f"Número de filas inconsistentes después de corregir: {len(inconsistencias_restantes)}")

Número de filas inconsistentes después de corregir: 0


In [56]:
# 1. Ver cuántos duplicados totales existen en las órdenes
duplicados_orders = orders.duplicated().sum()
print(f"Registros completamente duplicados en orders: {duplicados_orders}")

# 2. Si existen, los eliminamos dejando solo la primera aparición
if duplicados_orders > 0:
    orders = orders.drop_duplicates()
    print("¡Duplicados eliminados con éxito!")

Registros completamente duplicados en orders: 100
¡Duplicados eliminados con éxito!


In [57]:
# 1. Ver qué categorías únicas tenemos actualmente en orders
print("Categorías antes de corregir:")
print(orders['categoria_producto'].unique())

# 2. Reemplazamos 'Electronica' por 'Electrónica' para que coincida perfectamente con el catálogo
orders['categoria_producto'] = orders['categoria_producto'].replace('Electronica', 'Electrónica')

# 3. Validamos que el cambio se haya hecho con éxito
print("\nCategorías después de corregir:")
print(orders['categoria_producto'].unique())

Categorías antes de corregir:
['Moda' 'Electronica' 'Hogar' nan]

Categorías después de corregir:
['Moda' 'Electrónica' 'Hogar' nan]


In [58]:
# 1. Rellenar nulos en las columnas de texto/categorías
columnas_texto = ['pais', 'dispositivo', 'fuente_referencia', 'nombre_producto', 'categoria_producto']
orders[columnas_texto] = orders[columnas_texto].fillna('Desconocido')

# 2. Eliminar filas donde falten datos numéricos esenciales para las finanzas
orders = orders.dropna(subset=['cantidad', 'precio_unitario', 'monto_descuento'])

# 3. Comprobar que ya NO quedan nulos en orders
print("Valores nulos restantes por columna:")
print(orders.isnull().sum())

Valores nulos restantes por columna:
id_pedido             0
id_usuario            0
fecha_hora_pedido     0
pais                  0
dispositivo           0
fuente_referencia     0
nombre_producto       0
categoria_producto    0
cantidad              0
precio_unitario       0
monto_descuento       0
monto_total           0
dtype: int64


📦 Exportación: Una vez finalizada la limpieza, se exportan los datasets para utilizarlos en la última etapa del proyecto.

In [59]:
# exportar datasets
orders.to_csv('orders_clean.csv', index=False)
catalog.to_csv('catalog_clean.csv', index=False)
marketing.to_csv('marketing_clean.csv', index=False)

---

## 🔹 Paso 2: Analizar si el negocio es rentable

### 2.1 Cálculo de KPIs principales

**🎯 Objetivo:** Calcular los indicadores clave del negocio para evaluar ingresos, costos y rentabilidad.

Se usan los 3 datasets (`orders`, `catalog`, `marketing_spend`):

**📊 Parte 1: Rentabilidad del negocio**
- ¿Cuál es el ingreso total (revenue)? 
- ¿Cuál es el costo total? 
- ¿Cuánto se ha invertido en marketing? 
- ¿El negocio es rentable? (calcular profit)  

---

**📈 Parte 2: Comportamiento de ventas**
- ¿Cuál es el ticket promedio por orden? 
- ¿Cuál es la cantidad promedio de productos por orden? 
- ¿Cuál es el producto más vendido?
- ¿Cuánto se ha gastado en marketing por canal? 

In [60]:
# tu código aquí
# 2. Unificar las tablas (Merge) para poder calcular los costos
orders_enriquecido = pd.merge(orders, catalog, on='nombre_producto', how='left')

# 3. Calcular las variables financieras base de la Parte 1
revenue_total = orders_enriquecido['monto_total'].sum()
costo_total_productos = (orders_enriquecido['cantidad'] * orders_enriquecido['costo_unitario']).sum()
marketing_total = marketing['gasto'].sum()

# 4. Calcular la utilidad final (Profit) descontando costos de producto y marketing
profit = revenue_total - costo_total_productos - marketing_total

# 5. Mostrar los resultados en pantalla
print(f"Ingreso Total (Revenue): ${revenue_total:,.2f}")
print(f"Costo Total de Productos: ${costo_total_productos:,.2f}")
print(f"Inversión en Marketing: ${marketing_total:,.2f}")
print(f"Utilidad Neta (Profit): ${profit:,.2f}")


Ingreso Total (Revenue): $51,966,982.37
Costo Total de Productos: $43,124,069.01
Inversión en Marketing: $2,871,843.53
Utilidad Neta (Profit): $5,971,069.83


In [61]:
# 1. Ticket promedio por orden (promedio de la columna monto_total)
ticket_promedio = orders_enriquecido['monto_total'].mean()

# 2. Cantidad promedio de productos por orden
cantidad_promedio = orders_enriquecido['cantidad'].mean()

# 3. Producto más vendido (el que aparece más veces o suma más cantidad)
producto_mas_vendido = orders_enriquecido.groupby('nombre_producto')['cantidad'].sum().idxmax()
cant_max = orders_enriquecido.groupby('nombre_producto')['cantidad'].sum().max()

# 4. Gasto en marketing por canal
gasto_por_canal = marketing.groupby('canal')['gasto'].sum()

# Mostrar resultados
print(f"Ticket promedio por orden: ${ticket_promedio:,.2f}")
print(f"Cantidad promedio de productos por orden: {cantidad_promedio:.2f} unidades")
print(f"Producto más vendido: {producto_mas_vendido} ({cant_max:,.0f} unidades)")
print("\nGasto en marketing por canal:")
print(gasto_por_canal.apply(lambda x: f"${x:,.2f}").to_string())

Ticket promedio por orden: $2,083.18
Cantidad promedio de productos por orden: 7.12 unidades
Producto más vendido: Laptop-Gaming-16GB (144,198 unidades)

Gasto en marketing por canal:
canal
organic        $913,533.01
paid_search    $863,088.21
social         $918,043.21


---

## 🔹 Paso 3: Entender dónde se pierden los usuarios (funnel de conversión)

**🎯 Objetivo:** Analizar el comportamiento de los usuarios para identificar en qué etapa del proceso se pierden.


⚙️**Conexión a la base de datos**:  
Se ejecuta la línea de configuración para conectar con la base de datos y aplicar consultas SQL en la tabla **events**.

---

**📊 Parte 1: Construcción del funnel**
- ¿Cuántos usuarios llegan a cada etapa del funnel?  
- Se calcula el número de usuarios únicos por `nombre_evento`  
- Se ordenan los eventos según el flujo del usuario  

---

**📉 Parte 2: Análisis de conversión**
- Se calcula la tasa de conversión entre cada paso del funnel  
- Se identifica en qué etapa se pierde la mayor cantidad de usuarios  
- ¿Cuál es la tasa de conversión final?
---

In [62]:
import pandas as pd
from sqlalchemy import create_engine

# ======================
# Conexión (NO modificar)
# ======================
db_config = {
    'user': 'practicum_student',
    'pwd': 'QnmDH8Sc2TQLvy2G3Vvh7',
    'host': 'yp-trainers-practicum.cluster-czs0gxyx2d8w.us-east-1.rds.amazonaws.com',
    'port': 5432,
    'db': 'data-analyst-production-db-en'
}

connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(
    db_config['user'],
    db_config['pwd'],
    db_config['host'],
    db_config['port'],
    db_config['db']
)

engine = create_engine(connection_string, connect_args={'sslmode':'require'})

In [63]:
# Explorar tabla events
# =========================
query_events = '''
SELECT *
FROM events;
'''
events = pd.read_sql(query_events, con=engine)
events.head()

,id_usuario,id_sesion,nombre_evento,timestamp_evento,pais,dispositivo,fuente_referencia,categoria_producto
0,user_6772,6a97f2af-32ae-4186-8c92-04025be1a27b,first_visit,2025-05-17,Colombia,desktop,organic,Moda
1,user_5883,369b767c-1c33-4b2f-a652-c7c0ef92cfc9,add_to_cart,2025-02-23,Mexico,mobile,social,Hogar
2,user_5946,60039041-e78b-474c-87b3-c0b7e9c30708,add_payment_info,2025-05-15,Colombia,desktop,social,Electronica
3,user_827,18252a64-f389-4ef7-9e58-dadad4a3491e,purchase,2025-03-31,Mexico,mobile,social,Moda
4,user_2361,221b364e-cdc5-4668-b698-18d5ba849a67,first_visit,2025-01-22,Argentina,desktop,paid_search,Electronica


In [64]:
# PARTE 1: Totales del funnel
# ======================

query_totals = '''
SELECT 
    nombre_evento,
    COUNT(DISTINCT id_usuario) AS usuarios_unicos
FROM events
GROUP BY nombre_evento
ORDER BY 
    CASE nombre_evento
        WHEN 'first_visit' THEN 1
        WHEN 'select_item' THEN 2
        WHEN 'begin_checkout' THEN 3
        WHEN 'add_to_cart' THEN 4
        WHEN 'add_payment_info' THEN 5
        WHEN 'purchase' THEN 6
        ELSE 7
    END;
'''

totals = pd.read_sql(query_totals, con=engine)
totals

,nombre_evento,usuarios_unicos
0,first_visit,7796
1,select_item,7582
2,begin_checkout,7208
3,add_to_cart,7634
4,add_payment_info,6250
5,purchase,6240


In [65]:
# PARTE 2: Conversiones
# ======================

query_conversion = '''
WITH total_funnel AS (
    SELECT 
        nombre_evento,
        COUNT(DISTINCT id_usuario) AS usuarios_unicos,
        CASE nombre_evento
            WHEN 'first_visit' THEN 1
            WHEN 'select_item' THEN 2
            WHEN 'begin_checkout' THEN 3
            WHEN 'add_to_cart' THEN 4
            WHEN 'add_payment_info' THEN 5
            WHEN 'purchase' THEN 6
            ELSE 7
        END AS paso
    FROM events
    GROUP BY nombre_evento
)
SELECT 
    nombre_evento,
    usuarios_unicos,
    ROUND(100.0 * usuarios_unicos / LAG(usuarios_unicos, 1) OVER(ORDER BY paso), 2) AS tasa_conversion_paso,
    ROUND(100.0 * usuarios_unicos / FIRST_VALUE(usuarios_unicos) OVER(ORDER BY paso), 2) AS tasa_conversion_final
FROM total_funnel
ORDER BY paso;'''

conversion = pd.read_sql(query_conversion, con=engine)
conversion

,nombre_evento,usuarios_unicos,tasa_conversion_paso,tasa_conversion_final
0,first_visit,7796,NaN,100.00
1,select_item,7582,97.26,97.26
2,begin_checkout,7208,95.07,92.46
3,add_to_cart,7634,105.91,97.92
4,add_payment_info,6250,81.87,80.17
5,purchase,6240,99.84,80.04


In [66]:
# Identificar el paso con mayor caída (ignorando el primer NaN y el 105% anómalo)
conversion_filtrada = conversion[conversion['tasa_conversion_paso'] < 100]
peor_paso = conversion_filtrada.loc[conversion_filtrada['tasa_conversion_paso'].idxmin()]

print(f"Paso con mayor pérdida de usuarios: {peor_paso['nombre_evento']}")
print(f"Tasa de conversión en ese paso: {peor_paso['tasa_conversion_paso']}%")
print(f"Tasa de conversión final (purchase/first_visit): {conversion.iloc[-1]['tasa_conversion_final']}%")

Paso con mayor pérdida de usuarios: add_payment_info
Tasa de conversión en ese paso: 81.87%
Tasa de conversión final (purchase/first_visit): 80.04%


---

## 🔹 Paso 4: Evaluar si los usuarios regresan (retención por cohortes)

**🎯 Objetivo:** Analizar la retención de usuarios para entender si regresan después de registrarse.

**Tablas**

- `users` 
- `user_activity` 

---
1. Se identifica la cohorte de cada usuario según el **mes de registro**.


2. Se calcula la retención semanal: cuántos usuarios **se mantienen activos** en cada semana desde su registro.
   - `retenido_w1`: usuarios activos en la semana 1  
   - `retenido_w2`: usuarios activos en la semana 2  
   - `retenido_w3`: usuarios activos en la semana 3  


3. Se calcula el porcentaje de retención para cada semana, dividiendo los usuarios retenidos entre los clientes iniciales de la cohorte:  
   - `semana_1`: porcentaje de usuarios retenidos en la semana 1  
   - `semana_2`: porcentaje de usuarios retenidos en la semana 2  
   - `semana_3`: porcentaje de usuarios retenidos en la semana 3  

Se revisa que la columna de fecha esté en formato correcto (`DATE`).  
Se realiza la conversión usando: `CAST(fecha_registro AS DATE)`

In [67]:
# Explorar tabla users
# =========================
query_users = '''
SELECT *
FROM users;
'''
users = pd.read_sql(query_users, con=engine)
users.head(3)

,id_usuario,fecha_registro,país,dispositivo,tipo_plan
0,user_0,2025-01-29,Mexico,mobile,free
1,user_1,2025-01-07,Mexico,mobile,free
2,user_2,2025-03-12,Argentina,mobile,free


In [68]:
# Explorar tabla user_activity
# =========================
query_user_activity = '''
SELECT *
FROM user_activity;
'''
user_activity = pd.read_sql(query_user_activity, con=engine)
user_activity.head(3)

,id_usuario,fecha_actividad,dias_despues_registro,activo
0,user_0,2025-02-05,7,0
1,user_0,2025-02-12,14,1
2,user_0,2025-02-19,21,1


In [69]:
# Retención por cohortes
# ======================

query_cohort_retention_final = '''
WITH cohort_sizes AS (
    SELECT 
        TO_CHAR(CAST(fecha_registro AS DATE), 'YYYY-MM') AS cohorte_mes,
        COUNT(DISTINCT id_usuario) AS usuarios_iniciales
    FROM users
    GROUP BY 1
),
retention_counts AS (
    SELECT 
        TO_CHAR(CAST(u.fecha_registro AS DATE), 'YYYY-MM') AS cohorte_mes,
        COUNT(DISTINCT CASE WHEN ua.dias_despues_registro = 7 AND ua.activo = 1 THEN ua.id_usuario END) AS retenido_w1,
        COUNT(DISTINCT CASE WHEN ua.dias_despues_registro = 14 AND ua.activo = 1 THEN ua.id_usuario END) AS retenido_w2,
        COUNT(DISTINCT CASE WHEN ua.dias_despues_registro = 21 AND ua.activo = 1 THEN ua.id_usuario END) AS retenido_w3
    FROM users u
    LEFT JOIN user_activity ua ON u.id_usuario = ua.id_usuario
    GROUP BY 1
)
SELECT 
    c.cohorte_mes,
    c.usuarios_iniciales,
    ROUND(100.0 * r.retenido_w1 / c.usuarios_iniciales, 2) AS semana_1,
    ROUND(100.0 * r.retenido_w2 / c.usuarios_iniciales, 2) AS semana_2,
    ROUND(100.0 * r.retenido_w3 / c.usuarios_iniciales, 2) AS semana_3
FROM cohort_sizes c
JOIN retention_counts r ON c.cohorte_mes = r.cohorte_mes
ORDER BY c.cohorte_mes;'''

# Ejecutar la consulta
cohorte_final = pd.read_sql(query_cohort_retention_final, con=engine)
cohorte_final

,cohorte_mes,usuarios_iniciales,semana_1,semana_2,semana_3
0,2025-01,1627,42.84,41.06,40.32
1,2025-02,1444,42.31,42.17,43.98
2,2025-03,1636,41.38,43.09,42.18
3,2025-04,1606,42.34,43.40,41.28
4,2025-05,1687,41.20,40.07,41.85


In [70]:
# Promedio de retención por semana
print("Retención promedio semana 1:", round(cohorte_final['semana_1'].mean(), 2), "%")
print("Retención promedio semana 2:", round(cohorte_final['semana_2'].mean(), 2), "%")
print("Retención promedio semana 3:", round(cohorte_final['semana_3'].mean(), 2), "%")

Retención promedio semana 1: 42.01 %
Retención promedio semana 2: 41.96 %
Retención promedio semana 3: 41.92 %


---

## 🔹 Paso 5: Validar si los cambios generan impacto (test estadístico)

🎯 **Objetivo:** Evaluar si la modificación en la UI del checkout impacta la **tasa de conversión de compra**.

---

1. **Analizar el dataset** `experiment_checkout_ui.csv` para identificar la métrica principal **conversion**.
   - La métrica **conversion** es 1 si el usuario completó la compra, 0 si no.    
2. **Plantear la hipótesis estadística**     
3. **Aplicar el test estadístico adecuado** 
4. **Interpretar el resultado**  

---
Hipótesis estadística
   - **H₀ (Hipótesis nula):**  La modificación en la UI del checkout no genera impacto en la tasa de conversión. Las tasas de conversión del grupo de control (diseño viejo) y del grupo experimental (diseño nuevo) son estadísticamente iguales ($p_{control} = p_{experimental}$).
     
   - **H₁ (Hipótesis alternativa):** La modificación en la UI del checkout sí genera impacto en la tasa de conversión. Las tasas de conversión del grupo de control y del grupo experimental son estadísticamente diferentes ($p_{control} \neq p_{experimental}$).
   
**Test estadístico:** Prueba de proporciones Z (Z-test para dos proporciones), ya que estamos comparando una variable categórica binaria (conversión: 1 si compró, 0 si no) entre dos grupos independientes.

**Nivel de significancia alpha:** 0.05 (el estándar del 5% para validar significancia estadística).

In [71]:
# tu código aquí

import pandas as pd
from statsmodels.stats.proportion import proportions_ztest

# 1. Cargar el dataset desde la ruta correcta
df_experiment = pd.read_csv('datasets/experiment_checkout_ui.csv')

# 2. Calcular las métricas agregadas por variante
summary = df_experiment.groupby('variante').agg(
    total_usuarios=('id_usuario', 'count'),
    compras=('convirtio', 'sum')
).reset_index()

summary['conversion_rate'] = summary['compras'] / summary['total_usuarios']
print("Resumen Real del Experimento:")
print(summary)
print("-" * 60)

# 3. Preparar los datos para el Z-test
# Extraemos los éxitos (compras) y observaciones (totales) ordenados por grupo
conversions = summary['compras'].tolist()
nobs = summary['total_usuarios'].tolist()

# 4. Ejecutar el test estadístico de dos colas
z_stat, p_value = proportions_ztest(count=conversions, nobs=nobs, alternative='two-sided')

print(f"Estadístico Z: {z_stat:.4f}")
print(f"Valor p (p-value): {p_value:.6f}")

Resumen Real del Experimento:
      variante  total_usuarios  compras  conversion_rate
0      control            4965      779         0.156898
1  tratamiento            5035      820         0.162860
------------------------------------------------------------
Estadístico Z: -0.8133
Valor p (p-value): 0.416059


In [72]:
# Interpretación del resultado
alpha = 0.05
if p_value < alpha:
    print("Se rechaza H0: La diferencia ES estadísticamente significativa.")
    print("El nuevo diseño de checkout SÍ impacta la tasa de conversión.")
else:
    print("No se rechaza H0: La diferencia NO es estadísticamente significativa.")
    print(f"p-value ({p_value:.4f}) > alpha ({alpha}). No hay evidencia suficiente.")
    print("El nuevo diseño de checkout NO demuestra impacto significativo.")

No se rechaza H0: La diferencia NO es estadísticamente significativa.
p-value (0.4161) > alpha (0.05). No hay evidencia suficiente.
El nuevo diseño de checkout NO demuestra impacto significativo.


In [73]:
print(orders['monto_total'].describe())
print("Suma total:", orders['monto_total'].sum())

count    2.494600e+04
mean     2.083179e+03
std      9.925482e+04
min      5.240000e+00
25%      1.806400e+02
50%      3.416100e+02
75%      5.184450e+02
max      8.840200e+06
Name: monto_total, dtype: float64
Suma total: 51966982.37000001


In [74]:
print(orders['monto_total'].max())
print(orders.head(3)[['cantidad', 'precio_unitario', 'monto_descuento', 'monto_total']])

8840200.0
   cantidad  precio_unitario  monto_descuento  monto_total
0       2.0           332.69              0.0       665.38
1       1.0           176.86              5.0       171.86
2       2.0           102.99             10.0       195.98


In [76]:
orders.to_csv('orders_clean.csv', index=False)
print("Exportado correctamente")
print("Suma monto_total:", orders['monto_total'].sum())

Exportado correctamente
Suma monto_total: 51966982.37000001


In [77]:
print(orders.dtypes)
print(orders['monto_total'].head(10))


id_pedido                     object
id_usuario                    object
fecha_hora_pedido     datetime64[ns]
pais                          object
dispositivo                   object
fuente_referencia             object
nombre_producto               object
categoria_producto            object
cantidad                     float64
precio_unitario              float64
monto_descuento              float64
monto_total                  float64
dtype: object
0    665.38
1    171.86
2    195.98
3    242.87
4    336.28
5    179.34
6    327.18
7    747.36
8    949.54
9    334.30
Name: monto_total, dtype: float64


In [78]:
orders_export = orders.copy()
orders_export['fecha_hora_pedido'] = orders_export['fecha_hora_pedido'].dt.strftime('%Y-%m-%d')
orders_export.to_csv('orders_clean.csv', index=False)
print("Exportado correctamente")
print(orders_export.dtypes)

Exportado correctamente
id_pedido              object
id_usuario             object
fecha_hora_pedido      object
pais                   object
dispositivo            object
fuente_referencia      object
nombre_producto        object
categoria_producto     object
cantidad              float64
precio_unitario       float64
monto_descuento       float64
monto_total           float64
dtype: object


---

## 🔹 Paso 6: Comunicar los resultados (Dashboard en BI)

🎯 **Objetivo**:  
Crear un dashboard que muestre de manera clara y visual los resultados del análisis de ventas, costos, marketing y conversión. 

Se usarán los CSVs limpios del Paso 1:

- `orders_clean.csv`  
- `catalog_clean.csv`  
- `marketing_clean.csv`

---

1️⃣ Preparación de los datos
1. Cargar los CSVs en Power BI o Tableau.
2. Revisar relaciones:
   - `orders.nombre_producto` → `catalog.nombre_producto`
   - `orders.fecha_pedido` → tabla de fechas (crear calendario para análisis temporal)
   - `orders.fecha_pedido` → `dim_fecha.date`
3. Crear columnas calculadas necesarias
4. Crear tabla de fechas para poder calcular comparaciones YTD, YoY o períodos anteriores (`Previous Year`, `Previous Month`).

---

2️⃣ Dashboard 1: Overview Ejecutivo
**KPIs principales a mostrar:**
- Revenue total
- Profit total
- Gasto total en marketing
- Ticket promedio
- Cantidad promedio de productos por orden

**Visualizaciones sugeridas:**
- Tarjetas KPI para revenue, profit y gasto marketing
- Gráfico de líneas: evolución mensual de revenue o profit
- Gráfico de líneas YTD
- Gráfico de barras: revenue y profit por producto o categoría

---

 3️⃣ Dashboard 2: Detalle / Drill-through  
**Objetivo:** Permitir explorar los datos desde el KPI general hasta cada orden o producto.

**Visualizaciones sugeridas:**
- Tabla detallada de órdenes con:
  - producto, cantidad, revenue, cost, profit
  - color condicional (profit negativo en rojo, positivo en verde)
- Gráfico de barras por producto con medida `cantidad vendida`
- Drill-through: seleccionar un producto y ver todos los pedidos relacionados
- Filtros por fecha, categoría de producto, etc

---

## 🚀 Entrega Final

Comparte el acceso a tu Dashboard para revisión.   
Puedes entregar el Dashboard utilizando **Power BI o Tableau**.

Incluye **uno de los siguientes**:

- 🔗 Link público del dashboard publicado en **Power BI Service o Tableau Public / Tableau Cloud**
- 🔗 Link de **Google Drive o OneDrive** con el archivo del proyecto (`.pbix`) y los 3 csvs limpios.


### 📎 Enlace del Dashboard

In [ ]:
# (Pega aquí tu link)
# link de power bi o tableau
# link de one drive / google drive